In [12]:
!pip install --upgrade binance-historical-data plotly -q

In [13]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [14]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data

In [17]:
tickers = ['PAXGUSDT','BTCUSDT','PAXGBTC']
data_frequency = '1d'
data = []

# Download and store data
for ticker in tickers:
    download_ticker(ticker, "2022-01-01", "2025-07-25", data_frequency)
    ticker_data = process_csv_files(ticker, data_frequency)
    if ticker_data is not None: # Add this check
        ticker_data = ticker_data.drop(columns=['High','Low','close_time',  'quote_asset_volume',   'number_of_trades',     'taker_buy_base_asset_volume',  'taker_buy_quote_asset_volume', 'ignore'])
        ticker_data = ticker_data.rename(columns={
                'Open': f'{ticker}_open',
                #'High': f'{ticker}_high',
                #'Low': f'{ticker}_low',
                'Close': f'{ticker}_close',
                'volume': f'{ticker}_volume'
            })
        data.append(ticker_data)

# Merge list of
df1 = data[0]
df2 = data[1]
# df3 = data[2]
data = pd.merge(df1, df2, on='open_time', how='outer')
# data = pd.merge(data, df3, on='open_time', how='outer')
data.set_index('open_time', inplace=True)
data

---> Found overall tickers: 592
---> Filter to asked tickers: 8
------> Tickers left: 3
Download full data for 3 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20220101
---> End Date: 20250725


Tickers:   0%|          | 0/3 [00:00<?, ?it/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

...

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 0 months 0 days
---> For BTCUSD new data saved for: 0 months 0 days
---> Found overall tickers: 592
---> Filter to asked tickers: 7
------> Tickers left: 0
Download full data for 0 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20220101
---> End Date: 20250725


Tickers: 0it [00:00, ?it/s]

Tried to dump data for 0 tickers:
❗ Không có file CSV nào cho PAXGBTC


,PAXGUSDT_open,PAXGUSDT_close,PAXGUSDT_volume,BTCUSDT_open,BTCUSDT_close,BTCUSDT_volume
open_time,,,,,,
2022-01-02,1837.00,1835.00,930.4303,47722.66,47286.18,18340.46040
2022-01-03,1836.00,1809.00,1596.5055,47286.18,46446.10,27662.07710
2022-01-04,1810.00,1817.00,1308.9776,46446.10,45832.01,35491.41360
2022-01-05,1818.00,1816.00,3280.6306,45832.01,43451.13,51784.11857
2022-01-06,1816.00,1797.00,7057.4330,43451.14,43082.31,38880.37305
...,...,...,...,...,...,...
2025-06-26,3352.99,3332.97,4827.4522,107340.59,106947.06,10573.27279
2025-06-27,3333.00,3296.15,7379.5357,106947.06,107047.59,12232.44042
2025-06-28,3296.20,3289.58,1301.8120,107047.58,107296.79,3282.17352


In [ ]:
data['PAXG_per_BTC'] = data['PAXGUSDT_open'] / data['BTCUSDT_open']
data['PAXG/BTC_high']
data

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=data.index, y=data['PAXGUSDT_close'], mode='lines', name='PAXGUSDT Close'), secondary_y=False)
fig.add_trace(go.Scatter(x=data.index, y=data['BTCUSDT_close'], mode='lines', name='BTCUSDT Close'), secondary_y=False)
# fig.add_trace(go.Scatter(x=data.index, y=data['PAXGBTC_close'], mode='lines', name='PAXGBTC Close'), secondary_y=True)

fig.update_layout(
    title='Closing Prices of PAXGUSDT, BTCUSDT, and PAXGBTC',
    xaxis_title='Date',
    hovermode='x unified'
)

fig.update_yaxes(title_text="Price (USDT)", secondary_y=False)
fig.update_yaxes(title_text="Price (BTC)", secondary_y=True)

fig.show()